# 01 — Fetch Teams & Standings

Populates the `teams` and `standings` tables from `/v1/standings/now`.

In [3]:
# Mount Drive and import the shared helper module
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT_DIR = '/content/drive/MyDrive/NHL_Analytics_Hub'
sys.path.append(PROJECT_DIR)
import nhl_api_utils as nhl

DB_PATH = f'{PROJECT_DIR}/data/nhl_hockey.db'
SCHEMA_PATH = f'{PROJECT_DIR}/schema.sql'


Mounted at /content/drive


In [4]:
# conn = nhl.get_connection(DB_PATH)
# nhl.create_schema(conn, SCHEMA_PATH)


import os

# Ensure the 'data' directory exists before connecting
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

conn = nhl.get_connection(DB_PATH)
nhl.create_schema(conn, SCHEMA_PATH)
print("Database connected and schema created successfully!")

Schema created / verified.
Database connected and schema created successfully!


## Step 1 — Inspect the raw response

Always look at the real JSON before writing parsing code — field names can differ from what you expect.

In [5]:
raw = nhl.fetch_standings()
print(type(raw), list(raw.keys()))
raw['standings'][0]   # inspect one team's record


<class 'dict'> ['wildCardIndicator', 'standingsDateTimeUtc', 'standings']


{'clinchIndicator': 'p',
 'conferenceAbbrev': 'W',
 'conferenceHomeSequence': 1,
 'conferenceL10Sequence': 2,
 'conferenceName': 'Western',
 'conferenceRoadSequence': 1,
 'conferenceSequence': 1,
 'date': '2026-04-17',
 'divisionAbbrev': 'C',
 'divisionHomeSequence': 1,
 'divisionL10Sequence': 1,
 'divisionName': 'Central',
 'divisionRoadSequence': 1,
 'divisionSequence': 1,
 'gameTypeId': 2,
 'gamesPlayed': 82,
 'goalDifferential': 99,
 'goalDifferentialPctg': 1.207317,
 'goalAgainst': 203,
 'goalFor': 302,
 'goalsForPctg': 3.682927,
 'homeGamesPlayed': 41,
 'homeGoalDifferential': 49,
 'homeGoalsAgainst': 108,
 'homeGoalsFor': 157,
 'homeLosses': 9,
 'homeOtLosses': 6,
 'homePoints': 58,
 'homeRegulationPlusOtWins': 25,
 'homeRegulationWins': 25,
 'homeTies': 0,
 'homeWins': 26,
 'l10GamesPlayed': 10,
 'l10GoalDifferential': 14,
 'l10GoalsAgainst': 20,
 'l10GoalsFor': 34,
 'l10Losses': 2,
 'l10OtLosses': 1,
 'l10Points': 15,
 'l10RegulationPlusOtWins': 6,
 'l10RegulationWins': 6,
 'l

## Step 2 — Parse into `teams` and `standings` rows and load

In [7]:
teams_loaded = 0
standings_loaded = 0

for t in raw['standings']:
    team_abbrev = t['teamAbbrev']['default']
    team_row = {
        'team_abbrev': team_abbrev,
        'team_name': t['teamName']['default'],
        'conference_name': t.get('conferenceName'),
        'division_name': t.get('divisionName'),
        'logo_url': t.get('teamLogo'),
    }
    nhl.insert_ignore(conn, 'teams', team_row)
    teams_loaded += 1

conn.commit()
print(f'{teams_loaded} teams processed')


32 teams processed


In [8]:
for t in raw['standings']:
    team_abbrev = t['teamAbbrev']['default']
    team_id = nhl.get_team_id(conn, team_abbrev)

    standing_row = {
        'team_id': team_id,
        'season': str(t.get('seasonId')),
        'games_played': t.get('gamesPlayed'),
        'wins': t.get('wins'),
        'losses': t.get('losses'),
        'ot_losses': t.get('otLosses'),
        'points': t.get('points'),
        'goals_for': t.get('goalFor'),
        'goals_against': t.get('goalAgainst'),
        'home_wins': t.get('homeWins'),
        'away_wins': t.get('roadWins'),
        'streak_type': t.get('streakCode'),
        'streak_count': t.get('streakCount'),
    }
    nhl.insert_ignore(conn, 'standings', standing_row)
    standings_loaded += 1

conn.commit()
print(f'{standings_loaded} standings rows inserted')


32 standings rows inserted


In [9]:
# Quick sanity check
import pandas as pd
pd.read_sql('SELECT * FROM teams LIMIT 5', conn)


,team_id,team_abbrev,team_name,conference_name,division_name,logo_url
0,1,COL,Colorado Avalanche,Western,Central,https://assets.nhle.com/logos/nhl/svg/COL_ligh...
1,2,CAR,Carolina Hurricanes,Eastern,Metropolitan,https://assets.nhle.com/logos/nhl/svg/CAR_ligh...
2,3,DAL,Dallas Stars,Western,Central,https://assets.nhle.com/logos/nhl/svg/DAL_ligh...
3,4,BUF,Buffalo Sabres,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/BUF_ligh...
4,5,TBL,Tampa Bay Lightning,Eastern,Atlantic,https://assets.nhle.com/logos/nhl/svg/TBL_ligh...


In [8]:
pd.read_sql('SELECT * FROM standings LIMIT 5', conn)
conn.close()


In [26]:
SCHEMA_PATH = f'{PROJECT_DIR}/schema.sql'

updated_schema = """CREATE TABLE IF NOT EXISTS teams (
    team_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    team_abbrev     TEXT UNIQUE NOT NULL,
    team_name       TEXT NOT NULL,
    conference_name TEXT,
    division_name   TEXT,
    logo_url        TEXT
);

CREATE TABLE IF NOT EXISTS standings (
    standing_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    team_id       INTEGER NOT NULL,
    season        TEXT,
    games_played  INTEGER,
    wins          INTEGER,
    losses        INTEGER,
    ot_losses     INTEGER,
    points        INTEGER,
    goals_for     INTEGER,
    goals_against INTEGER,
    home_wins     INTEGER,
    away_wins     INTEGER,
    streak_type   TEXT,
    streak_count  INTEGER,
    FOREIGN KEY (team_id) REFERENCES teams(team_id),
    UNIQUE (team_id, season)
);

CREATE TABLE IF NOT EXISTS players (
    player_id       INTEGER PRIMARY KEY,
    team_id         INTEGER NOT NULL,
    first_name      TEXT,
    last_name       TEXT,
    position        TEXT,
    jersey_number   INTEGER,
    birth_date      TEXT,
    birth_country   TEXT,
    height_cm       REAL,
    weight_kg       REAL,
    shoots_catches  TEXT,
    headshot_url    TEXT,
    FOREIGN KEY (team_id) REFERENCES teams(team_id)
);

CREATE TABLE IF NOT EXISTS games (
    game_id         INTEGER PRIMARY KEY,
    season          TEXT,
    game_type       INTEGER,
    game_date       TEXT,
    home_team_id    INTEGER,
    away_team_id    INTEGER,
    home_score      INTEGER,
    away_score      INTEGER,
    game_state      TEXT,
    venue_name      TEXT,
    FOREIGN KEY (home_team_id) REFERENCES teams(team_id),
    FOREIGN KEY (away_team_id) REFERENCES teams(team_id)
);

CREATE TABLE IF NOT EXISTS game_stats (
    stat_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    game_id         INTEGER NOT NULL,
    player_id       INTEGER NOT NULL,
    team_id         INTEGER NOT NULL,
    goals           INTEGER,
    assists         INTEGER,
    points          INTEGER,
    shots_on_goal   INTEGER,
    penalty_min     INTEGER,
    toi             TEXT,
    plus_minus      INTEGER,
    FOREIGN KEY (game_id)   REFERENCES games(game_id),
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id)   REFERENCES teams(team_id),
    UNIQUE (game_id, player_id)
);

CREATE TABLE IF NOT EXISTS skater_season_stats (
    stat_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    player_id       INTEGER NOT NULL,
    season          TEXT,
    team_id         INTEGER,
    games_played    INTEGER,
    goals           INTEGER,
    assists         INTEGER,
    points          INTEGER,
    plus_minus      INTEGER,
    penalty_min     INTEGER,
    shots           INTEGER,
    avg_toi         TEXT,
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id)   REFERENCES teams(team_id),
    UNIQUE (player_id, season)
);

CREATE TABLE IF NOT EXISTS goalie_season_stats (
    stat_id              INTEGER PRIMARY KEY AUTOINCREMENT,
    player_id            INTEGER NOT NULL,
    season               TEXT,
    team_id              INTEGER,
    games_played         INTEGER,
    wins                 INTEGER,
    losses               INTEGER,
    ot_losses            INTEGER,
    save_pct             REAL,
    goals_against_avg    REAL,
    shutouts             INTEGER,
    saves                INTEGER,
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id)   REFERENCES teams(team_id),
    UNIQUE (player_id, season)
);
"""

with open(SCHEMA_PATH, 'w') as f:
    f.write(updated_schema)
print("schema.sql updated with the missing comma!")

schema.sql updated with the missing comma!


In [27]:
import os

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print("Old database deleted.")

conn = nhl.get_connection(DB_PATH)
nhl.create_schema(conn, SCHEMA_PATH)

Old database deleted.
Schema created / verified.


In [11]:
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM standings;")
print("Actual rows in standings table:", cursor.fetchone()[0])

Actual rows in standings table: 32


In [12]:
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM teams;")
print("Teams:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM standings;")
print("Standings:", cursor.fetchone()[0])

Teams: 32
Standings: 32


In [13]:
cursor.execute("""
SELECT
    t.team_abbrev,
    t.team_name,
    s.season,
    s.games_played,
    s.wins,
    s.losses,
    s.points
FROM teams t
JOIN standings s
    ON t.team_id = s.team_id
ORDER BY s.points DESC;
""")

rows = cursor.fetchall()

for row in rows:
    print(row)

('COL', 'Colorado Avalanche', '20252026', 82, 55, 16, 121)
('CAR', 'Carolina Hurricanes', '20252026', 82, 53, 22, 113)
('DAL', 'Dallas Stars', '20252026', 82, 50, 20, 112)
('BUF', 'Buffalo Sabres', '20252026', 82, 50, 23, 109)
('TBL', 'Tampa Bay Lightning', '20252026', 82, 50, 26, 106)
('MTL', 'Montréal Canadiens', '20252026', 82, 48, 24, 106)
('MIN', 'Minnesota Wild', '20252026', 82, 46, 24, 104)
('BOS', 'Boston Bruins', '20252026', 82, 45, 27, 100)
('OTT', 'Ottawa Senators', '20252026', 82, 44, 27, 99)
('PIT', 'Pittsburgh Penguins', '20252026', 82, 41, 25, 98)
('PHI', 'Philadelphia Flyers', '20252026', 82, 43, 27, 98)
('WSH', 'Washington Capitals', '20252026', 82, 43, 30, 95)
('VGK', 'Vegas Golden Knights', '20252026', 82, 39, 26, 95)
('EDM', 'Edmonton Oilers', '20252026', 82, 41, 30, 93)
('UTA', 'Utah Mammoth', '20252026', 82, 43, 33, 92)
('DET', 'Detroit Red Wings', '20252026', 82, 41, 31, 92)
('CBJ', 'Columbus Blue Jackets', '20252026', 82, 40, 30, 92)
('ANA', 'Anaheim Ducks', '20

In [14]:
# 1. NULL team_id
cursor.execute("""
SELECT COUNT(*)
FROM standings
WHERE team_id IS NULL;
""")
print("NULL team_id:", cursor.fetchone()[0])


# 2. Orphan standings
cursor.execute("""
SELECT COUNT(*)
FROM standings s
LEFT JOIN teams t
    ON s.team_id = t.team_id
WHERE t.team_id IS NULL;
""")
print("Orphan standings:", cursor.fetchone()[0])


# 3. Duplicate team + season
cursor.execute("""
SELECT team_id, season, COUNT(*)
FROM standings
GROUP BY team_id, season
HAVING COUNT(*) > 1;
""")

duplicates = cursor.fetchall()
print("Duplicate (team_id, season) groups:", len(duplicates))

NULL team_id: 0
Orphan standings: 0
Duplicate (team_id, season) groups: 0
